In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))

from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT
from SIDER_dataset.libraries.feature_evaluation_methods import feature_variance_reduction_scores
from SIDER_dataset.libraries.utils import get_clus_path
%load_ext autoreload
%autoreload 2

In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
paths

18 datasets


[{'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv',
  'dataset_name': 'CPI+fingerprint_cardiac',
  'label_set': ['se_C0016382',
   'se_C0018799',
   'se_C0003811',
   'se_C0428977',
   'se_C0027051',
   'se_C0018790'],
  'features': 2147,
  'original_features': ['cpi_9606.ENSP00000000442',
   'cpi_9606.ENSP00000001008',
   'cpi_9606.ENSP00000003084',
   'cpi_9606.ENSP00000003100',
   'cpi_9606.ENSP00000005178',
   'cpi_9606.ENSP00000011292',
   'cpi_9606.ENSP00000011653',
   'cpi_9606.ENSP00000012443',
   'cpi_9606.ENSP00000013034',
   'cpi_9606.ENSP00000014930',
   'cpi_9606.ENSP00000019103',
   'cpi_9606.ENSP00000023897',
   'cpi_9606.ENSP00000039007',
   'cpi_9606.ENSP00000044462',
   'cpi_9606.ENSP00000054668',
   'cpi_9606.ENSP00000078429',
   'cpi_9606.ENSP00000155840',
   'cpi_9606.ENSP00000164139',
   'cpi_9606.ENSP00000171757',
   'cpi_9606.ENSP00000176183',
   'cpi_9606.ENSP00000176195',
 

In [3]:
paths = [path for path in paths if "ten_mid" in path["dataset_name"]]
len(paths)

3

In [4]:
k = 10
random_state = 42
performances = []
ranking_criteria = "MDI"  # or "VAR"
include_original_features_options = [True, False]
training_algorithm = "Variance Reduction"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]
max_size = 5
cv_results = []

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criteria}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            training_algorithm,
            "_".join(eval_criteria),
            str(max_size),
        ]
    )
    logging_path = f"XofN_filter/logs/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        if ranking_criteria == "VAR":
            feature_rankings = feature_variance_reduction_scores(train_dataset[features],
                                                                 train_dataset[path["label_set"]])
        elif ranking_criteria == "MDI":
            feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])
        else:
            raise NotImplementedError

        XofN_groupings, avg_features, gen_XofN_time = generate_XofN_list_multi_custom(
            train_dataset,
            feature_rankings,
            max_size,
            path["label_set"],
            logger
        )
        if len(XofN_groupings) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = group_features(
                    train_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_test_dataset = group_features(
                    test_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_train_dataset.to_csv(f"XofN_filter/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"XofN_filter/tmp/test_dataset.csv", index=False)

                training_start = time.perf_counter()
                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "XofN_filter/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"XofN_filter/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            eval_criteria + ['nodes', 'leaves', 'groups',
                             'avg_group_features', 'gen_XofN_time', 'training_time']].mean().reset_index()
        print(averages)
        cv_results.append(averages)
        performances = []
# paths[0] - features[:10] desktop 0.29m
# laptop ??m
# desktop 14h


--- Running with label:'['se_C0009676', 'se_C0041657', 'se_C0002994', 'se_C0042571', 'se_C0004604', 'se_C0041834', 'se_C0085631', 'se_C0040822', 'se_C0042373', 'se_C0021053']' training_algorithm:'Variance Reduction' eval_criterion:'['averageAUROC', 'HammingLoss', 'SubsetAccuracy', 'RankingLoss', 'MacroPrecision', 'MacroRecall', 'MacroFOne']' max_size:'5' ---
Logs can be found in XofN_filter/logs/CPI+fingerprint_ten_mid_Variance Reduction_averageAUROC_HammingLoss_SubsetAccuracy_RankingLoss_MacroPrecision_MacroRecall_MacroFOne_5_logs.txt.

Fold 1/10 (CPI+fingerprint_ten_mid 1/3)

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 2147/2147 [12:26<00:00,  2.88feat/s] 


XofN groups with 2 features: 30
XofN groups with 3 features: 7
XofN groups with 4 features: 9
XofN groups with 5 features: 402
pruning: True, include_original_features: with_org, averageAUROC: 0.5436044, HammingLoss: 0.38273, SubsetAccuracy: 0.10791, RankingLoss: 0.38177, MacroPrecision: 0.45909, MacroRecall: 0.27702, MacroFOne: 0.33737, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5453187, HammingLoss: 0.44317, SubsetAccuracy: 0.05036, RankingLoss: 0.35356, MacroPrecision: 0.40498, MacroRecall: 0.47027, MacroFOne: 0.4328, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5454955, HammingLoss: 0.36978, SubsetAccuracy: 0.1223, RankingLoss: 0.38399, MacroPrecision: 0.48176, MacroRecall: 0.28304, MacroFOne: 0.35102, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5458158, HammingLoss: 0.44388, SubsetAccuracy: 0.0071942, RankingLoss: 0.35265, MacroPrecision: 0.39906, MacroRecall: 0.45944, MacroFOne: 0.42606, 

Fold 2/10 (CPI+fin

🔄 Processing features: 100%|██████████| 2147/2147 [12:30<00:00,  2.86feat/s]


XofN groups with 2 features: 13
XofN groups with 3 features: 9
XofN groups with 4 features: 8
XofN groups with 5 features: 405
pruning: True, include_original_features: with_org, averageAUROC: 0.6426293, HammingLoss: 0.31942, SubsetAccuracy: 0.15108, RankingLoss: 0.34349, MacroPrecision: 0.5504, MacroRecall: 0.38107, MacroFOne: 0.44923, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6006948, HammingLoss: 0.38849, SubsetAccuracy: 0.071942, RankingLoss: 0.33417, MacroPrecision: 0.44246, MacroRecall: 0.5154, MacroFOne: 0.47571, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5792706, HammingLoss: 0.30791, SubsetAccuracy: 0.15108, RankingLoss: 0.36345, MacroPrecision: 0.63648, MacroRecall: 0.24135, MacroFOne: 0.34919, 
pruning: False, include_original_features: no_org, averageAUROC: 0.6115845, HammingLoss: 0.37698, SubsetAccuracy: 0.071942, RankingLoss: 0.32971, MacroPrecision: 0.45803, MacroRecall: 0.53908, MacroFOne: 0.49407, 

Fold 3/10 (CPI+fin

...

XofN groups with 2 features: 3
XofN groups with 3 features: 4
XofN groups with 4 features: 4
XofN groups with 5 features: 101
pruning: True, include_original_features: with_org, averageAUROC: 0.5804445, HammingLoss: 0.33893, SubsetAccuracy: 0.1374, RankingLoss: 0.32782, MacroPrecision: 0.46641, MacroRecall: 0.28607, MacroFOne: 0.35207, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5745698, HammingLoss: 0.42672, SubsetAccuracy: 0.045802, RankingLoss: 0.34089, MacroPrecision: 0.38391, MacroRecall: 0.50355, MacroFOne: 0.43433, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5920313, HammingLoss: 0.35267, SubsetAccuracy: 0.14504, RankingLoss: 0.33647, MacroPrecision: 0.4596, MacroRecall: 0.38231, MacroFOne: 0.41516, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5922055, HammingLoss: 0.43206, SubsetAccuracy: 0.061069, RankingLoss: 0.31854, MacroPrecision: 0.39098, MacroRecall: 0.55487, MacroFOne: 0.45719, 

Fold 9/10 (fingerpr

🔄 Processing features: 100%|██████████| 540/540 [01:35<00:00,  5.64feat/s]


XofN groups with 2 features: 2
XofN groups with 3 features: 3
XofN groups with 4 features: 1
XofN groups with 5 features: 104
pruning: True, include_original_features: with_org, averageAUROC: 0.5666192, HammingLoss: 0.37786, SubsetAccuracy: 0.1145, RankingLoss: 0.33035, MacroPrecision: 0.43078, MacroRecall: 0.24859, MacroFOne: 0.30874, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5877531, HammingLoss: 0.41603, SubsetAccuracy: 0.038168, RankingLoss: 0.35275, MacroPrecision: 0.43493, MacroRecall: 0.51552, MacroFOne: 0.46769, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5448063, HammingLoss: 0.3916, SubsetAccuracy: 0.10687, RankingLoss: 0.35997, MacroPrecision: 0.4182, MacroRecall: 0.23596, MacroFOne: 0.29944, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5496489, HammingLoss: 0.44809, SubsetAccuracy: 0.038168, RankingLoss: 0.385, MacroPrecision: 0.39125, MacroRecall: 0.45681, MacroFOne: 0.41952, 

Fold 10/10 (fingerprin

🔄 Processing features: 100%|██████████| 540/540 [01:36<00:00,  5.62feat/s]


XofN groups with 3 features: 7
XofN groups with 4 features: 3
XofN groups with 5 features: 101
pruning: True, include_original_features: with_org, averageAUROC: 0.5576617, HammingLoss: 0.38244, SubsetAccuracy: 0.12977, RankingLoss: 0.3433, MacroPrecision: 0.53388, MacroRecall: 0.19998, MacroFOne: 0.28738, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5890007, HammingLoss: 0.40534, SubsetAccuracy: 0.068702, RankingLoss: 0.33757, MacroPrecision: 0.48637, MacroRecall: 0.47317, MacroFOne: 0.4785, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5507356, HammingLoss: 0.38855, SubsetAccuracy: 0.14504, RankingLoss: 0.35128, MacroPrecision: 0.52155, MacroRecall: 0.18102, MacroFOne: 0.26547, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5664737, HammingLoss: 0.43817, SubsetAccuracy: 0.068702, RankingLoss: 0.33784, MacroPrecision: 0.44796, MacroRecall: 0.48986, MacroFOne: 0.46717, 
   pruning include_original_features              d

In [5]:
# All results
save_path = "XofN_filter/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_ten_mid,0.563495,0.430429,0.048063,0.356595,0.402927,0.488958,0.439039,980.2,490.6,110.8,4.860297,96.060877,0.700176
1,False,with_org,fingerprint_ten_mid,0.573206,0.419311,0.045778,0.346055,0.416316,0.493034,0.447313,981.8,491.4,110.8,4.860297,96.060877,1.349122
2,True,no_org,fingerprint_ten_mid,0.558789,0.366436,0.123583,0.352629,0.453489,0.233259,0.299101,89.4,45.2,110.8,4.860297,96.060877,0.700176
3,True,with_org,fingerprint_ten_mid,0.573596,0.359570,0.114440,0.340774,0.473784,0.251221,0.320821,95.6,48.3,110.8,4.860297,96.060877,1.349122
4,False,no_org,CPI_ten_mid,0.570970,0.419172,0.047846,0.367079,0.435220,0.486643,0.456122,773.6,387.3,329.6,4.787968,429.850277,0.896050
5,False,with_org,CPI_ten_mid,0.584062,0.415154,0.041531,0.353514,0.440773,0.498586,0.464208,801.8,401.4,329.6,4.787968,429.850277,2.680915
6,True,no_org,CPI_ten_mid,0.589400,0.353755,0.118369,0.365819,0.532128,0.285609,0.361335,61.0,31.0,329.6,4.787968,429.850277,0.896050
7,True,with_org,CPI_ten_mid,0.593718,0.351383,0.115666,0.360931,0.529604,0.313409,0.385733,78.4,39.7,329.6,4.787968,429.850277,2.680915
8,False,no_org,CPI+fingerprint_ten_mid,0.577341,0.408734,0.047597,0.347622,0.424922,0.493686,0.454009,1022.8,511.9,438.4,4.828520,697.756258,1.222673
9,False,with_org,CPI+fingerprint_ten_mid,0.578968,0.404366,0.051913,0.351211,0.427563,0.477883,0.448103,1019.6,510.3,438.4,4.828520,697.756258,3.740729


In [6]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_filter/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, eval_criteria, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
2,no_org,fingerprint_ten_mid,0.559,0.366,0.124,0.353,0.453,0.233,0.299,89.4; 45.2,110.8; 4.9,1.479105,96.1; 0.7
3,with_org,fingerprint_ten_mid,0.574,0.360,0.114,0.341,0.474,0.251,0.321,95.6; 48.3,110.8; 4.9,1.479105,96.1; 1.3
6,no_org,CPI_ten_mid,0.589,0.354,0.118,0.366,0.532,0.286,0.361,61.0; 31.0,329.6; 4.8,28.885798,429.9; 0.9
7,with_org,CPI_ten_mid,0.594,0.351,0.116,0.361,0.530,0.313,0.386,78.4; 39.7,329.6; 4.8,28.885798,429.9; 2.7
10,no_org,CPI+fingerprint_ten_mid,0.593,0.338,0.131,0.346,0.525,0.267,0.349,82.2; 41.6,438.4; 4.8,30.176840,697.8; 1.2
11,with_org,CPI+fingerprint_ten_mid,0.602,0.343,0.132,0.346,0.510,0.286,0.360,96.2; 48.6,438.4; 4.8,30.176840,697.8; 3.7
